## Notebook — Sync Supabase → Fabric (questionnaire SPV / projets)

### Objectif
Répliquer dans le Lakehouse Fabric (`lakehouse_gold`) les tables liées au **questionnaire** et au **périmètre SPV**, pour les exploiter ensuite dans Power BI / notebooks d'analyse :

| Source Supabase                  | Cible Lakehouse (Delta)               | Volume approx. |
|----------------------------------|---------------------------------------|----------------|
| `project_questionnaire`          | `spv_questionnaire_answers`           | ~ 940 lignes   |
| `questionnaire_field_definitions`| `questionnaire_field_definitions`     | ~ 100 lignes   |
| `spv_affaires`                   | `spv_affaires`                        | ~ 40 lignes    |

### Direction
Supabase (source) → Fabric Lakehouse (cible). Mode **snapshot complet** : chaque exécution remplace les tables Delta (`overwrite` + `overwriteSchema=true`).

### Connexion (alignée sur `nb_divalto_mouvements_all_sync`)
Les variables `SUPABASE_URL` et `SUPABASE_SERVICE_ROLE_KEY` sont injectées par le **pipeline Fabric** via les Base parameters (`@variables('SUPABASE_URL')`, `@variables('SUPABASE_SERVICE_ROLE_KEY')`). Le bloc `notebookutils.variableLibrary.getLibrary("env-temp")` est laissé **commenté** car il déclenche un `431 Request Header Fields Too Large` en exécution interactive.

**Clé `service_role` obligatoire** : la RLS de `project_questionnaire` appelle `has_role(auth.uid(), ...)` qui crashe avec `auth.uid() = NULL` (anon role) → `500 Internal Server Error`. Le `service_role` contourne entièrement la RLS.

### Pré-requis Fabric
- Lakehouse `lakehouse_gold` attaché au notebook.
- Pipeline Fabric configuré avec `SUPABASE_URL` + `SUPABASE_SERVICE_ROLE_KEY` en Base parameters.

In [ ]:
# ─── 0) Config ──────────────────────────────────────────────────
# Variables injectées par le pipeline Fabric (Base parameters).
# Le bloc variableLibrary reste commenté (bug Fabric 431 en interactif).
#
# Pour exécuter en INTERACTIF (hors pipeline) : décommente les 3 lignes
# variableLibrary ci-dessous, ou définis manuellement les 2 variables
# juste avant cette cellule.
#
# env = notebookutils.variableLibrary.getLibrary("env-temp")
# SUPABASE_URL              = env.SUPABASE_URL
# SUPABASE_SERVICE_ROLE_KEY = env.SUPABASE_SERVICE_ROLE_KEY

import requests
import pandas as pd
from pyspark.sql import functions as F

GOLD_DB   = "lakehouse_gold"
PAGE_SIZE = 1000  # PostgREST default max

print("[0] Config OK —", SUPABASE_URL)

In [ ]:
# ─── 1) Helper REST paginé (PostgREST) ─────────────────────────────────
# Auth = service_role : bypass RLS (indispensable pour project_questionnaire
# qui a une policy SELECT cassée pour auth.uid() NULL).
def supabase_select(table: str, select: str = "*", filters: dict | None = None):
    """Lecture paginée d'une table via PostgREST.
    - filters : dict optionnel {col: 'op.val'} ex. {'updated_at': 'gt.2026-01-01'}
    Retourne une liste de dicts (peut être vide).
    """
    out, offset = [], 0
    base = f"{SUPABASE_URL.rstrip('/')}/rest/v1/{table}"
    params = [("select", select)]
    for k, v in (filters or {}).items():
        params.append((k, v))
    while True:
        h = {
            "apikey":        SUPABASE_SERVICE_ROLE_KEY,
            "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
            "Range-Unit":    "items",
            "Range":         f"{offset}-{offset + PAGE_SIZE - 1}",
        }
        r = requests.get(base, headers=h, params=params, timeout=60)
        r.raise_for_status()
        chunk = r.json()
        out.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
    return out

print("[1] Helper REST prêt (auth = service_role, bypass RLS)")

In [ ]:
# ─── 2) Sync générique : Supabase → Delta (overwrite + schema evolution) ────
TS_COLS_DEFAULT = ("created_at", "updated_at")

def sync_table(src: str, dst: str, select: str = "*", ts_cols=TS_COLS_DEFAULT) -> int:
    rows = supabase_select(src, select)
    print(f"  ← {src:<40s} : {len(rows):>6d} lignes lues")
    if not rows:
        print(f"  ⚠️  Aucune donnée — {GOLD_DB}.{dst} laissée telle quelle.")
        return 0
    pdf = pd.DataFrame(rows)
    # JSONB → string (Delta n'aime pas les dicts hétérogènes via createDataFrame)
    import json
    for col in pdf.columns:
        if pdf[col].map(lambda x: isinstance(x, (dict,))).any():
            pdf[col] = pdf[col].map(lambda v: json.dumps(v) if isinstance(v, (dict, list)) else v)
    sdf = spark.createDataFrame(pdf)
    for c in ts_cols:
        if c in sdf.columns:
            sdf = sdf.withColumn(c, F.to_timestamp(c))
    n = sdf.count()
    (sdf.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_DB}.{dst}"))
    print(f"  → {GOLD_DB}.{dst:<40s} : {n:>6d} lignes écrites")
    return n

print("[2] Fonction sync_table prête")

In [ ]:
# ─── 3) Exécution ─────────────────────────────────────────────────
print("=" * 70)
print("  Sync Supabase → Fabric : questionnaire SPV")
print("=" * 70)

total = 0
total += sync_table("project_questionnaire",           "spv_questionnaire_answers")
total += sync_table("questionnaire_field_definitions", "questionnaire_field_definitions")
total += sync_table("spv_affaires",                    "spv_affaires")

print("=" * 70)
print(f"  Terminé — {total} lignes au total")
print("=" * 70)

In [ ]:
# ─── 4) (Optionnel) Vue dénormalisée prête à l'usage BI ──────────────────
# Joint les réponses avec leur définition de champ pour avoir le label,
# le type de champ et l'ordre dans une seule table Power BI-friendly.
df_ans = spark.table(f"{GOLD_DB}.spv_questionnaire_answers")
df_def = spark.table(f"{GOLD_DB}.questionnaire_field_definitions")

df_enriched = (
    df_ans.alias("a")
          .join(
              df_def.select(
                  F.col("champ_id").alias("def_champ_id"),
                  F.col("label").alias("champ_label"),
                  F.col("type_champ").alias("def_type_champ"),
                  F.col("has_evaluation_risque"),
                  F.col("required"),
                  F.col("order_index"),
              ).alias("d"),
              F.col("a.champ_id") == F.col("d.def_champ_id"),
              "left",
          )
          .drop("def_champ_id")
)

n = df_enriched.count()
(df_enriched.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.spv_questionnaire_enriched"))
print(f"  → {GOLD_DB}.spv_questionnaire_enriched : {n} lignes")

### Notes

- **Sécurité** : ce notebook utilise la clé `service_role` (bypass RLS). Il doit donc être exécuté uniquement dans un workspace Fabric de confiance, jamais commité avec les valeurs en dur. Les variables passent par le pipeline.
- **Évolution du schéma** : `overwriteSchema=true` absorbe l'ajout/suppression de colonnes côté Supabase. Pour un changement de type (`text` → `jsonb`), drop manuel de la table Delta.
- **Incrémental** : si volumétrie > 50k lignes, passer en MERGE sur `updated_at` (helper supporte déjà `filters={"updated_at": f"gt.{last_run_iso}"}`).
- **Vérification post-run** :
  ```sql
  SELECT 'answers'  AS t, COUNT(*) FROM lakehouse_gold.spv_questionnaire_answers
  UNION ALL SELECT 'defs',     COUNT(*) FROM lakehouse_gold.questionnaire_field_definitions
  UNION ALL SELECT 'affaires', COUNT(*) FROM lakehouse_gold.spv_affaires
  UNION ALL SELECT 'enriched', COUNT(*) FROM lakehouse_gold.spv_questionnaire_enriched;
  ```